<a href="https://colab.research.google.com/github/sabyapaul/AgenticAI-Lab/blob/main/Agentic_AI_Bronze_to_Silver_Eligibility_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install packages**

In [ ]:
!pip install pyspark gradio openai -q

**Importing Open AI Key**

In [ ]:
import os
from getpass import getpass
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

**Importing Python packages panda, gradio & openai**

In [ ]:
import gradio as gr
import pandas as pd
from datetime import datetime
from openai import OpenAI

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

spark = SparkSession.builder \
    .appName("Agentic_AI_Component_Based_Demo") \
    .getOrCreate()

**1. Memory / Knowledge Base**

In [ ]:
memory_knowledge_base = {
    "business_rule": "Claim service date must fall within member eligibility period.",
    "failure_code": "NO_ACTIVE_ELIGIBILITY",
    "agent_action": "FLAG_FOR_REVIEW",
    "valid_action": "LOAD_TO_SILVER",
    "domain": "Healthcare Medicare Claims"
}

**2. Tools**

In [ ]:
def load_bronze_claims():
    claims_data = [
        ("C001", "M001", "2026-01-15", 1200.00),
        ("C002", "M002", "2026-02-10", 800.00),
        ("C003", "M003", "2026-03-05", 500.00),
        ("C004", "M004", "2026-01-20", 700.00),
        ("C005", "M001", "2026-04-10", 300.00)
    ]

    return spark.createDataFrame(
        claims_data,
        ["claim_id", "member_id", "service_date", "claim_amount"]
    ).withColumn("service_date", col("service_date").cast("date"))


def load_bronze_eligibility():
    eligibility_data = [
        ("M001", "2026-01-01", "2026-03-31"),
        ("M002", "2026-01-01", "2026-12-31"),
        ("M003", "2026-04-01", "2026-12-31")
    ]

    return spark.createDataFrame(
        eligibility_data,
        ["member_id", "effective_date", "termination_date"]
    ).withColumn(
        "effective_date", col("effective_date").cast("date")
    ).withColumn(
        "termination_date", col("termination_date").cast("date")
    )


def validate_eligibility_tool():
    invalid_claims = spark.sql("""
        SELECT c.claim_id, c.member_id, c.service_date, c.claim_amount
        FROM silver_claim c
        LEFT JOIN silver_eligibility e
          ON c.member_id = e.member_id
         AND c.service_date BETWEEN e.effective_date AND e.termination_date
        WHERE e.member_id IS NULL
    """)

    valid_claims = spark.sql("""
        SELECT c.claim_id, c.member_id, c.service_date, c.claim_amount
        FROM silver_claim c
        INNER JOIN silver_eligibility e
          ON c.member_id = e.member_id
         AND c.service_date BETWEEN e.effective_date AND e.termination_date
    """)

    return valid_claims, invalid_claims


def flag_claims_tool(invalid_claims):
    return invalid_claims \
        .withColumn("validation_status", lit("FAILED")) \
        .withColumn("reason_code", lit(memory_knowledge_base["failure_code"])) \
        .withColumn("agent_action", lit(memory_knowledge_base["agent_action"]))


def load_to_silver_tool(valid_claims):
    return valid_claims \
        .withColumn("validation_status", lit("PASSED")) \
        .withColumn("reason_code", lit("")) \
        .withColumn("agent_action", lit(memory_knowledge_base["valid_action"]))

**3. Planning**

In [ ]:
def planning_agent():
    plan = [
        "Step 1: Load Bronze claims data",
        "Step 2: Load Bronze eligibility data",
        "Step 3: Register data as Silver staging views",
        "Step 4: Validate claim service date against eligibility period",
        "Step 5: Load valid claims to Silver",
        "Step 6: Flag invalid claims for review",
        "Step 7: Generate LLM explanation and audit summary"
    ]

    return "\n".join(plan)

**4. AI Model**

In [ ]:
def ai_model_reasoning(final_df, exception_df, plan):
    prompt = f"""
You are an Agentic AI healthcare data quality agent.

Agentic AI Components:
1. AI Model: OpenAI model for reasoning and explanation
2. Tools: PySpark validation tools
3. Memory/Knowledge Base: Business rules and reason codes
4. Planning: Step-by-step Bronze to Silver validation workflow

Business Rule:
{memory_knowledge_base["business_rule"]}

Execution Plan:
{plan}

Final Silver Output:
{final_df.to_string(index=False)}

Exception Queue:
{exception_df.to_string(index=False)}

Generate a simple client-facing explanation:
- What the agent checked
- What decisions it made
- What actions it took
- Why failed claims were flagged
- Business value
"""

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    return response.output_text

**5. Main Agent**

In [ ]:
class EligibilityValidationAgent:

    def __init__(self):
        self.agent_name = "Eligibility Validation Agent"
        self.memory = memory_knowledge_base

    def run(self):

        audit = []

        audit.append(f"{datetime.now()} - Agent started")

        plan = planning_agent()
        audit.append(f"{datetime.now()} - Planning completed")

        bronze_claim = load_bronze_claims()
        bronze_eligibility = load_bronze_eligibility()
        audit.append(f"{datetime.now()} - Bronze data loaded")

        bronze_claim.createOrReplaceTempView("silver_claim")
        bronze_eligibility.createOrReplaceTempView("silver_eligibility")
        audit.append(f"{datetime.now()} - Silver staging views created")

        valid_claims, invalid_claims = validate_eligibility_tool()
        audit.append(f"{datetime.now()} - Eligibility validation tool executed")

        passed_claims = load_to_silver_tool(valid_claims)
        failed_claims = flag_claims_tool(invalid_claims)
        audit.append(f"{datetime.now()} - Agent actions completed")

        final_df = passed_claims.unionByName(failed_claims)

        final_pdf = final_df.toPandas()
        exception_pdf = failed_claims.toPandas()

        llm_explanation = ai_model_reasoning(
            final_pdf,
            exception_pdf,
            plan
        )

        audit.append(f"{datetime.now()} - OpenAI model generated reasoning")

        return plan, final_pdf, exception_pdf, llm_explanation, "\n".join(audit)

**6. Gradio UI**

In [ ]:
def run_demo():
    agent = EligibilityValidationAgent()
    return agent.run()


with gr.Blocks(title="Agentic AI Components Demo") as demo:

    gr.Markdown("""
# Agentic AI Components Demo

## Agentic AI = AI Model + Tools + Memory + Planning

### Use Case
Bronze to Silver Healthcare Claims Validation

### Business Rule
Claim service date must fall within member eligibility period.
""")

    run_button = gr.Button("Run Eligibility Validation Agent")

    plan_output = gr.Textbox(label="Planning Component", lines=8)

    final_table = gr.Dataframe(label="Silver Claims Output")

    exception_table = gr.Dataframe(label="Exception Queue")

    llm_reasoning = gr.Markdown(label="AI Model Reasoning")

    audit_output = gr.Textbox(label="Audit Trail", lines=10)

    run_button.click(
        fn=run_demo,
        inputs=[],
        outputs=[
            plan_output,
            final_table,
            exception_table,
            llm_reasoning,
            audit_output
        ]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a9b353f4e7472310ed.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
